# 02 — Silver transformation (OSS)

Map one manufacturing catalog option to long `tenant-metrics` Silver rows and assert the contract.


In [ ]:
from pathlib import Path
from ambient_pipeline.notebook_bootstrap import ensure_pipeline_on_path, apply_spark_tuning

ROOT = ensure_pipeline_on_path(Path.cwd())
assert ROOT is not None, "Run from ambient-core checkout (lib/ambient_pipeline missing)"
print(f"repo root: {ROOT}")


In [ ]:
from ambient_pipeline.perf import create_local_spark

spark = create_local_spark(app_name="ambient-oss-notebooks", shuffle_partitions=4)
apply_spark_tuning(spark)
print(spark.version)


In [ ]:
from datetime import datetime, timezone
from ambient_pipeline.bronze_catalog_map import bronze_to_tenant_metrics
from ambient_pipeline.catalog_loader import load_data_option
from ambient_pipeline.contracts import ContractLoader
from ambient_pipeline.storage_paths import resolve_table_path
from ambient_pipeline.validation import SilverValidator

ORG_ID = "demo-org"
RUN_ID = "notebook-silver-001"
OPTION = "Allmanufacturingds-inventory-records"
csv_path = ROOT / "data" / "raw" / f"{OPTION}.csv"
out_base = str(ROOT / ".lakehouse" / "demo")
silver_table = resolve_table_path("local", out_base, "demo", "silver", "tenant_metrics")

option = load_data_option(OPTION)
fields = [f["name"] if isinstance(f, dict) else str(f) for f in (option or {}).get("fields") or []]
mapping = {name: name for name in fields}

raw = spark.read.option("header", True).csv(str(csv_path))
stamped = bronze_to_tenant_metrics(
    raw,
    org_id=ORG_ID,
    mapping_json=mapping,
    catalog_option_key=OPTION,
    run_id=RUN_ID,
    source_type="csv_upload",
    source_path=str(csv_path),
    ingestion_ts=datetime.now(timezone.utc).isoformat(),
)

loader = ContractLoader()
contract = loader.load("tenant-metrics-v1.yaml")
loader.enforce_bronze_lineage(contract)
loader.assert_required_columns(set(stamped.columns), contract, "notebook_silver")

silver = SilverValidator(completeness_threshold=0.5).add_silver_provenance(
    stamped, RUN_ID, datetime.now(timezone.utc).isoformat()
)
(
    silver.write.format("delta").mode("overwrite").save(silver_table)
)
print(f"silver rows={spark.read.format('delta').load(silver_table).count()} path={silver_table}")
silver.select("name", "value", "industry", "metric_id").show(truncate=False)
